In [137]:
#Configuracion de entorno
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.config import Paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score


paths = Paths()

In [115]:
# Cargamos los datos limpios del noteboook anterior 
data = pd.read_parquet(paths.data_noOutlierData)
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total Price,Outlier Flag
0,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.60,0
1,489435,22353,LUNCHBOX WITH CUTLERY FAIRY CAKES,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.60,0
2,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,17.85,0
3,489436,84879,ASSORTED COLOUR BIRD ORNAMENT,16,2009-12-01 09:06:00,1.69,13078.0,United Kingdom,27.04,0
4,489436,22119,PEACE WOODEN BLOCK LETTERS,3,2009-12-01 09:06:00,6.95,13078.0,United Kingdom,20.85,0


In [116]:
def estimate_churn_days(data, quantile=0.75):
    max_date = data['InvoiceDate'].max()
    recency = (
        data
        .groupby('Customer ID')['InvoiceDate']
        .max()
        .apply(lambda x: (max_date - x).days)
    )
    return int(recency.quantile(quantile))

churnDays = estimate_churn_days(data, quantile=0.75)
churnDays


378

In [117]:
# Definimos fecha maxima del dataset
maxDate = data['InvoiceDate'].max()
refDate = maxDate - pd.Timedelta(days=churnDays)

In [118]:
historicalData = data[data['InvoiceDate'] <= refDate]
futureData = data[
    (data['InvoiceDate'] > refDate) &
    (data['InvoiceDate'] <= maxDate)
]

In [119]:
tempFuture = futureData.copy()

churnData = (
    tempFuture
        .groupby('Customer ID')
        .size()
        .reset_index(name='numPurchasesFuture')
)

churnData['churnFlag'] = (churnData['numPurchasesFuture'] == 0).astype(int)
churnData = churnData[['Customer ID','churnFlag']]
churnData.head()

,Customer ID,churnFlag
0,12347.0,0
1,12348.0,0
2,12349.0,0
3,12350.0,0
4,12351.0,0


In [120]:
RFM = (
    historicalData.groupby('Customer ID')
        .agg(
            Recency = ('InvoiceDate', lambda x: (refDate - x.max()).days),
            Frequency = ('Invoice', 'nunique'),
            Monetary =  ('Total Price', 'sum')
        )
        .reset_index()
)
RFM.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,150,10,327.86
1,12347.0,25,1,480.85
2,12348.0,59,1,101.20
3,12349.0,29,2,1373.44
4,12352.0,14,1,113.75


In [121]:
additionalFeatures = historicalData.copy()
additionalFeatures = (
    additionalFeatures.groupby('Customer ID')
        .agg(
            ticketPromedio = ('Total Price', 'mean'),
            uniqueProducts = ('StockCode', 'nunique'),
            daysActive =  ('InvoiceDate', lambda x: (x.max() - x.min()).days),
            averageDaysOfBuys = ('InvoiceDate', lambda x: x.sort_values().diff().dt.days.mean())
        )
        .reset_index()
)

additionalFeatures.dropna(inplace=True)
additionalFeatures.head()

,Customer ID,ticketPromedio,uniqueProducts,daysActive,averageDaysOfBuys
0,12346.0,10.245625,26,196,6.193548
1,12347.0,16.028333,30,0,0.000000
2,12348.0,12.650000,8,0,0.000000
3,12349.0,17.836883,72,181,2.381579
4,12352.0,22.750000,5,0,0.000000


In [122]:
historicalFeatures = pd.merge(RFM, additionalFeatures, how='left', on='Customer ID')
historicalFeatures.head()

,Customer ID,Recency,Frequency,Monetary,ticketPromedio,uniqueProducts,daysActive,averageDaysOfBuys
0,12346.0,150,10,327.86,10.245625,26.0,196.0,6.193548
1,12347.0,25,1,480.85,16.028333,30.0,0.0,0.000000
2,12348.0,59,1,101.20,12.650000,8.0,0.0,0.000000
3,12349.0,29,2,1373.44,17.836883,72.0,181.0,2.381579
4,12352.0,14,1,113.75,22.750000,5.0,0.0,0.000000


In [123]:
historicalChurn = pd.merge(historicalFeatures, churnData, how='left', on='Customer ID')
historicalChurn['churnFlag'] = (
    historicalChurn['churnFlag']
        .fillna(1)
        .astype(int)
)
historicalChurn.dropna(inplace=True)
historicalChurn.head()

,Customer ID,Recency,Frequency,Monetary,ticketPromedio,uniqueProducts,daysActive,averageDaysOfBuys,churnFlag
0,12346.0,150,10,327.86,10.245625,26.0,196.0,6.193548,1
1,12347.0,25,1,480.85,16.028333,30.0,0.0,0.000000,0
2,12348.0,59,1,101.20,12.650000,8.0,0.0,0.000000,0
3,12349.0,29,2,1373.44,17.836883,72.0,181.0,2.381579,0
4,12352.0,14,1,113.75,22.750000,5.0,0.0,0.000000,0


In [124]:
historicalChurn['churnFlag'].value_counts(normalize=True)

churnFlag
0    0.649809
1    0.350191
Name: proportion, dtype: float64

In [125]:
(historicalChurn['Customer ID'].nunique()) == (historicalFeatures['Customer ID'].nunique())

False

In [126]:
(historicalData['InvoiceDate'].max() <= refDate) & (futureData['InvoiceDate'].min() > refDate)

True

In [127]:
comparison = (
    historicalChurn
    .groupby('churnFlag')
    .agg({
        'Recency': 'mean',
        'Frequency': 'mean',
        'Monetary': 'mean',
        'ticketPromedio': 'mean',
        'daysActive': 'mean',
        'uniqueProducts': 'mean',
        'averageDaysOfBuys': 'mean'
    })
    .round(2)
)
comparison

,Recency,Frequency,Monetary,ticketPromedio,daysActive,uniqueProducts,averageDaysOfBuys
churnFlag,,,,,,,
0,62.17,5.61,1422.7,13.68,163.16,72.74,3.81
1,130.07,1.99,396.6,13.41,61.12,31.68,2.79


In [128]:
historicalChurn.head()

,Customer ID,Recency,Frequency,Monetary,ticketPromedio,uniqueProducts,daysActive,averageDaysOfBuys,churnFlag
0,12346.0,150,10,327.86,10.245625,26.0,196.0,6.193548,1
1,12347.0,25,1,480.85,16.028333,30.0,0.0,0.000000,0
2,12348.0,59,1,101.20,12.650000,8.0,0.0,0.000000,0
3,12349.0,29,2,1373.44,17.836883,72.0,181.0,2.381579,0
4,12352.0,14,1,113.75,22.750000,5.0,0.0,0.000000,0


In [129]:

y = historicalChurn['churnFlag']

In [130]:
transformedX = historicalChurn.copy()


transformedX['logUniqueProducts'] = np.log1p(transformedX['uniqueProducts'])
transformedX['logAverageDaysOfBuys'] = np.log1p(transformedX['averageDaysOfBuys'])
transformedX['logFrequency'] = np.log1p(transformedX['Frequency'])
transformedX['logMonetary'] = np.log1p(transformedX['Monetary'])

transformedX = transformedX[['Recency', 'logMonetary', 'logFrequency', 'ticketPromedio', 'daysActive', 'logAverageDaysOfBuys', 'logUniqueProducts']]
transformedX.head()

,Recency,logMonetary,logFrequency,ticketPromedio,daysActive,logAverageDaysOfBuys,logUniqueProducts
0,150,5.795632,2.397895,10.245625,196.0,1.973185,3.295837
1,25,6.177633,0.693147,16.028333,0.0,0.000000,3.433987
2,59,4.626932,0.693147,12.650000,0.0,0.000000,2.197225
3,29,7.225802,1.098612,17.836883,181.0,1.218343,4.290459
4,14,4.742756,0.693147,22.750000,0.0,0.000000,1.791759


In [131]:
Y =  historicalChurn['churnFlag']

In [174]:
X_train, X_test, y_train, y_test = train_test_split(
    transformedX,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

logit = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # importante para churn
    random_state=42
)

logit.fit(X_train, y_train)

y_pred_proba = logit.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
y_pred = logit.predict(X_test)

recall_churn = recall_score(y_test, y_pred, pos_label=1)
print(f"ROC AUC: {round(roc_auc,6)},\nRecall {round(recall_churn,6)}")

coef_df = pd.DataFrame({
    'Feature': ['Recency', 'logMonetary', 'logFrequency', 'ticketPromedio', 'daysActive', 'logAverageDaysOfBuys', 'logUniqueProducts'],
    'Coefficient': logit.coef_[0]
})

coef_df['Sign'] = np.where(coef_df['Coefficient'] > 0, 'Positive', 'Negative')

coef_df.sort_values(by='Coefficient', ascending=False)

ROC AUC: 0.79056,
Recall 0.736715


,Feature,Coefficient,Sign
5,logAverageDaysOfBuys,0.010883,Positive
0,Recency,0.003961,Positive
4,daysActive,-0.000922,Negative
3,ticketPromedio,-0.015191,Negative
6,logUniqueProducts,-0.165730,Negative
1,logMonetary,-0.210132,Negative
2,logFrequency,-0.918413,Negative


In [175]:
transformedX.corr(method='spearman')

,Recency,logMonetary,logFrequency,ticketPromedio,daysActive,logAverageDaysOfBuys,logUniqueProducts
Recency,1.000000,-0.432204,-0.459880,0.021336,-0.484959,-0.269156,-0.400402
logMonetary,-0.432204,1.000000,0.777936,0.047210,0.700190,0.330733,0.875385
logFrequency,-0.459880,0.777936,1.000000,0.021445,0.886967,0.640566,0.665651
ticketPromedio,0.021336,0.047210,0.021445,1.000000,0.040382,0.235568,-0.371755
daysActive,-0.484959,0.700190,0.886967,0.040382,1.000000,0.775731,0.597761
logAverageDaysOfBuys,-0.269156,0.330733,0.640566,0.235568,0.775731,1.000000,0.168997
logUniqueProducts,-0.400402,0.875385,0.665651,-0.371755,0.597761,0.168997,1.000000


In [178]:
transformedX2 = transformedX.copy()
variablesTry = ['Recency', 'logFrequency', 'logAverageDaysOfBuys', 'ticketPromedio']

transformedX2 = transformedX2[variablesTry]
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    transformedX2,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

logit2 = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # importante para churn
    random_state=42
)

logit2.fit(X_train2, y_train2)

y_pred_proba2 = logit2.predict_proba(X_test2)[:, 1]

roc_auc2 = roc_auc_score(y_test2, y_pred_proba2)
y_pred2 = logit2.predict(X_test2)

recall_churn2 = recall_score(y_test2, y_pred2, pos_label=1)
print(f"ROC AUC: {round(roc_auc2,6)},\nRecall {round(recall_churn2,6)}")

coef_df2 = pd.DataFrame({
    'Feature': variablesTry,
    'Coefficient': logit2.coef_[0]
})

coef_df2['Sign'] = np.where(coef_df2['Coefficient'] > 0, 'Positive', 'Negative')

coef_df2.sort_values(by='Coefficient', ascending=False)

ROC AUC: 0.788542,
Recall 0.736715


,Feature,Coefficient,Sign
2,logAverageDaysOfBuys,0.071706,Positive
0,Recency,0.004543,Positive
3,ticketPromedio,-0.007493,Negative
1,logFrequency,-1.594785,Negative
